# Normal-like CM network outputs

This notebook generates only the normal-like Top10 node-correlation heatmap.


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


BASE_DIR = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_epi_analysis")
OUTPUT_DIR = BASE_DIR / "balanced_joint_nmf_outputs"
JOINT_CM_DIR = OUTPUT_DIR / "joint_cm"
TABLE_DIR = JOINT_CM_DIR / "tables"
SHARED_DIR = OUTPUT_DIR / "shared"
FIGURE_DIR = JOINT_CM_DIR / "figures"

TOP10_NODE_TABLE = TABLE_DIR / "joint_cm_cell_subtype_nodes_top10_from_H_df.csv"
NORM_FREQUENCY_TABLE = SHARED_DIR / "non_epi_subtype_frequency_global_minmax.csv"
SAMPLE_STATUS_TABLE = SHARED_DIR / "sample_status.csv"

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"


def top10_nodes_by_cm() -> dict[str, list[str]]:
    top10 = pd.read_csv(TOP10_NODE_TABLE)
    top10 = top10.sort_values(["CM", "rank"])
    return top10.groupby("CM", sort=False)["node"].apply(list).to_dict()


def plot_status_node_correlation_heatmap(
    context_label: str,
    samples: pd.Index,
    norm_df: pd.DataFrame,
    top10_nodes: dict[str, list[str]],
    output_stem: Path,
) -> None:
    cm_names = list(top10_nodes)
    n_cols = 3
    n_rows = int(np.ceil(len(cm_names) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.2, n_rows * 4.15), squeeze=False)
    axes = axes.flatten()

    image = None
    for ax, cm_name in zip(axes, cm_names):
        nodes = [node for node in top10_nodes[cm_name] if node in norm_df.columns]
        ax.set_title(cm_name, fontsize=11, fontweight="bold")
        if len(nodes) < 2 or len(samples) < 4:
            ax.axis("off")
            ax.text(0.5, 0.5, "Insufficient data", ha="center", va="center", fontsize=9)
            continue

        corr = norm_df.loc[samples, nodes].corr(method="pearson").loc[nodes, nodes]
        image = ax.imshow(corr.to_numpy(dtype=float), cmap="RdBu_r", vmin=-1.0, vmax=1.0, aspect="equal")
        ax.set_xticks(np.arange(len(nodes)))
        ax.set_yticks(np.arange(len(nodes)))
        ax.set_xticklabels(nodes, rotation=90, ha="center", fontsize=6)
        ax.set_yticklabels(nodes, fontsize=6)
        ax.tick_params(length=0)
        for spine in ax.spines.values():
            spine.set_visible(False)

    for ax in axes[len(cm_names):]:
        ax.axis("off")

    if image is not None:
        cb = fig.colorbar(image, ax=axes.tolist(), shrink=0.55, pad=0.015)
        cb.set_label("Pearson correlation (r)", fontsize=10)
        cb.ax.tick_params(labelsize=8)

    fig.suptitle(
        f"{context_label} CM top10 node-node correlation heatmaps",
        y=1.01,
        fontweight="bold",
        fontsize=15,
    )
    fig.savefig(output_stem.with_suffix(".pdf"), bbox_inches="tight", dpi=300)
    fig.savefig(output_stem.with_suffix(".svg"), bbox_inches="tight", dpi=300)
    plt.close(fig)


def plot_normal_outputs() -> None:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    norm_df = pd.read_csv(NORM_FREQUENCY_TABLE, index_col=0)
    sample_status = pd.read_csv(SAMPLE_STATUS_TABLE, index_col=0)
    top10_nodes = top10_nodes_by_cm()
    normal_samples = sample_status.index[
        sample_status["status"].eq("normal-like")
    ].intersection(norm_df.index)

    output_stem = FIGURE_DIR / "normal_like_top10_node_correlation_heatmap_no_edge_filter"
    plot_status_node_correlation_heatmap(
        "Normal-like",
        normal_samples,
        norm_df,
        top10_nodes,
        output_stem,
    )
    print("Saved:", output_stem.with_suffix(".pdf"))
    print("Saved:", output_stem.with_suffix(".svg"))


plot_normal_outputs()
